In [318]:
import pandas as pd
import numpy as np

In [319]:
df1 = pd.read_csv('student-mat.csv', sep=';')
df2 = pd.read_csv('student-por.csv', sep=';')

df = pd.concat([df1, df2], ignore_index=True)
df = df.drop_duplicates(subset=["school","sex","age","address","famsize","Pstatus","Medu","Fedu","Mjob","Fjob","reason","nursery","internet"], keep='first')

In [320]:
n = len(df)
n_val = int(n*0.2)
n_test = int(n*0.2)
n_train = n - n_val - n_test

np.random.seed(2)
idx = np.arange(n)
np.random.shuffle(idx)

df_train = df.iloc[idx[:n_train]].reset_index(drop=True)
df_val = df.iloc[idx[n_train:n_train+n_val]].reset_index(drop=True)
df_test = df.iloc[idx[n_train+n_val:]].reset_index(drop=True)

In [321]:
y_train = np.log1p(df_train.G3.values)
y_val = np.log1p(df_val.G3.values)
y_test = np.log1p(df_test.G3.values)

del df_train['G3']
del df_val['G3']
del df_test['G3']

In [322]:
categories_variables = list(df_train.dtypes[df_train.dtypes=='string'].index)
print(categories_variables)
categories = {}
for c in categories_variables:
    categories[c] = list(df_train[c].value_counts().index)

['school', 'sex', 'address', 'famsize', 'Pstatus', 'Mjob', 'Fjob', 'reason', 'guardian', 'schoolsup', 'famsup', 'paid', 'activities', 'nursery', 'higher', 'internet', 'romantic']


In [323]:
base = df_train.select_dtypes(include='integer').columns.tolist()
base = [cat for cat in base if cat not in ['G1', 'G2']]

In [324]:
def prepare_X(df):
    df = df.copy()
    features = base.copy()

    for c, values in categories.items():
        for v in values:
            df['%s_%s' % (c,v)] = (df[c]==v).astype('int')
            features.append('%s_%s' % (c,v))

    X = df[features].values
    return X

In [325]:
def linear(X,y,r):
    ones = np.ones(X.shape[0])
    X = np.column_stack([ones, X])

    XTX = X.T.dot(X)
    XTX = XTX + r*np.eye(XTX.shape[0])

    XTX_inv = np.linalg.inv(XTX)
    w_full = XTX_inv.dot(X.T).dot(y)
    return w_full[0], w_full[1:]

In [326]:
def rmse(y, y_pred):
    err = (y-y_pred)**2
    mse = err.mean()
    return np.sqrt(mse)


In [327]:
X_train = prepare_X(df_train)
r = 100000
for v in [0, 0.001, 0.01, 0.1,1,10,100,1000]:
    w0, w = linear(X_train, y_train, v)
    y_pred = w0 + X_train.dot(w)
    score = rmse(y_train, y_pred)
    if score<r:
        r=score

In [328]:
X_train = prepare_X(df_train)

In [329]:
w0, w = linear(X_train, y_train, r)
y_pred = w0 + X_train.dot(w)
train_score = rmse(y_train, y_pred)

In [330]:
X_val = prepare_X(df_val)
y_pred = w0 + X_val.dot(w)
val_score = rmse(y_val, y_pred)

In [331]:
df_full_train = pd.concat([df_train, df_val])
X_full_train = prepare_X(df_full_train)
y_full_train = np.concatenate([y_train, y_val])
w0, w = linear(X_full_train, y_full_train, r)
y_pred = w0 + X_full_train.dot(w)
full_train_score = rmse(y_full_train, y_pred)

In [332]:
X_test = prepare_X(df_test)
y_pred = w0 + X_test.dot(w)
test_score = rmse(y_test, y_pred)


In [333]:
print('train_score: ', train_score)
print('val_score: ', val_score)
print('full_train_score: ', full_train_score)
print('test_score: ', test_score)
print(np.expm1(y_test[0]), np.expm1(y_pred[0]))

train_score:  0.649292307337247
val_score:  0.6006529733884658
full_train_score:  0.6290663136574639
test_score:  0.599846050951755
14.0 11.731271076816311
